# New Patient Starts (NPS) Features — All Cohorts

Cohort-based refactor of `05_claim_rx_nps_monthly` (NBA 3.0, Wenjie Chen). Sections 1–3 walk through every table the original uses and what it builds from them; the cohort refactor and validation follow in later sections.

Source: `/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/03_Feature_Engineering/03_HCP_Level_Features/05_claim_rx_nps_monthly`

## What this notebook does, in plain words

It counts, per HCP per month, how many patients **started** a drug class or brand with that HCP.

A "start" is the patient's **first ever Rx claim** for that drug class (or brand) in the data, written by **any** HCP. The HCP who wrote that first claim gets credit for the start.

| Feature | Question it answers | Column |
| --- | --- | --- |
| Patients visiting | How many distinct patients had an Rx claim with this HCP this month? (denominator for `PCT_`) | `TOTAL_PATIENTS_VISIT` |
| Starts per drug class | How many patients started each `TREATMENT_CATEGORY` with this HCP? | `TOTAL_PATIENTS_START_<CLASS>` (e.g. `..._SGLT2`) |
| Starts per brand | Same for 8 brand groups (INVOKANA 100/300MG, FARXIGA, JARDIANCE, SPIRONOLACTONE-MRA, Diuretic + MRA, DPP-IV, GLP-1) | `TOTAL_PATIENTS_START_<BRAND>` (e.g. `..._JARDIANCE`) |
| Total new patients | How many distinct patients started *any* drug class with this HCP? | `TOTAL_NEW_PATIENT_START` |

Then the usual steps: fill missing months, sum over the last 1/3/6/12 months, `RUNNING_` totals, `PCT_` shares (each start column / `TOTAL_PATIENTS_VISIT` in the same window) and write one row per provider per month to `kerendia_nba_3_0_claim_rx_nps_monthly`.

Unlike notebooks 01–04, the input table already carries `PROVIDER_ID`, so the original works at provider level from the start (no NPI to provider mapping step).

**How to read this notebook:** section 1 shows the input table; section 2 shows the claim-level tables the original builds (the step just before counting); section 3 shows the original output. Everything is shown for the 10 test HCPs (validation mode).

## 0. Connection and Config

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

### Load Libraries

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
msal 1.36.0 requires cryptography<49,>=2.5, but you have cryptography 50.0.2 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Get Data from Scope

[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



### Define Snowflake Connection Function
Choose appropiate below items based on the Brand
- Role
- Warehouse
- Schema
- Database

In [0]:
%run "../00_cohort_config"

# NBA 3.1 — Pipeline Configuration

Single source of truth for all parameters, cohort logic, and helper functions
used across the patient journey (NB01–NB06) and HCP feature engineering notebooks.

**Location:** `/kerendia_nba_3_1_develop_clone/00_cohort_config`  
**Usage:** `%run "../00_cohort_config"` from any subfolder notebook.

## Section 1 — Global Parameters
Shared across patient journey and HCP feature notebooks.

OFFSET_DAY       = 7
DATA_START_DATE  = 2023-01-01
COHORT_ORIGIN    = 2025-01-01
COHORT_LAST      = 2026-10-01
COHORT_MONTH     = 2025-02
COHORT_MONTHS    = all (complete data only)
VALIDATION_MODE  = True


## Section 2 — Cohort Functions
`assign_cohort_month()` maps dates to offset-aligned months. Used by both patient journey and HCP features.

22 cohorts defined (C1 – C22)


## Section 3 — Validation / Production Filters
Controlled by `VALIDATION_MODE` in Section 1.  
* `get_test_npis()` — returns the 10 NPI list or `None` (data source)  
* `apply_hcp_filter(df)` — patient-level filter (for patient journey notebooks: finds patients linked to test NPIs)  
* `apply_npi_filter(df)` — NPI-level filter (for HCP feature notebooks: filters directly on NPI column)  

Both filter functions no-op in production mode.

## Section 4 — HCP Feature Engineering Helpers
Reusable functions for the claims monthly notebooks (01–06).

## Section 5 — HCP Cohort Parameters
Lookback windows, cutoff date, and window bounds used by all HCP feature notebooks (01–06).  
Set `COHORT_MONTH` in Section 1. When `None`, this section is skipped (patient journey runs).

Cohort: 2025-02 | Offset: 7 days | Cutoff: 2025-03-06 | ID: 2025-02_offset7
  Last  1m: 2025-02-06 -> 2025-03-06  (AVG: 2025-01-06 -> 2025-03-06, /3)
  Last  3m: 2024-12-06 -> 2025-03-06  (AVG: 2024-11-06 -> 2025-03-06, /5)
  Last  6m: 2024-09-06 -> 2025-03-06  (AVG: 2024-08-06 -> 2025-03-06, /8)
  Last  9m: 2024-06-06 -> 2025-03-06  (AVG: 2024-05-06 -> 2025-03-06, /11)
  Last 12m: 2024-03-06 -> 2025-03-06  (AVG: 2024-02-06 -> 2025-03-06, /14)


Testing with current OFFSET_DAY = 7
-------------------------------------------------------
  2025-01-07 → 2025-01-01  (expected 2025-01-01)  [PASS]
  2025-01-06 → 2024-12-01  (expected 2024-12-01)  [PASS]
  2025-02-06 → 2025-01-01  (expected 2025-01-01)  [PASS]
  2025-02-07 → 2025-02-01  (expected 2025-02-01)  [PASS]
  2025-10-04 → 2025-09-01  (expected 2025-09-01)  [PASS]

HCP window vs cohort assignment (22 cohorts): PASS
  e.g. 2025-02: 1-month window (2025-02-06, 2025-03-06], cutoff 2025-03-06

✅ Config loaded. Available to all pipeline notebooks.


## 1. The input table (`kerendia_nba_3_0_Rx_05`)

The original reads a single table: one row per Rx claim, with the HCP as `PROVIDER_ID` (and `NPI_NUMBER`), the HCP's `PRIMARY_SPECIALTY_CODE`, the patient, the service date, `TREATMENT_CATEGORY` (drug class) and `CATEGORY` (brand group).

Two views are loaded:
* **`df_rx`**: claims written by the 10 test HCPs (what gets counted).
* **`df_rx_pat`**: *all* claims of those HCPs' patients, written by any HCP. The first-start date of a patient has to be looked up across every HCP, so this view is needed to reproduce the original exactly in validation mode.

The original keeps claims from 2019-11 onwards (cell 7). The table starts in 2021-01, so that filter removes nothing.

In [0]:
import builtins
_VOLUME = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
           "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/")
_QUERY_OUTPUT    = _VOLUME + "query_output/"
_NOTEBOOK_OUTPUT = _VOLUME + "notebook_output/"

_BRANDS = ['INVOKANA 100MG', 'INVOKANA 300MG', 'FARXIGA', 'JARDIANCE',
           'SPIRONOLACTONE-MRA', 'Diuretic + MRA', 'DPP-IV', 'GLP-1']

# Same cleaning and date filter as the original cell 7
df_rx_raw = spark.read.format("delta").load(_QUERY_OUTPUT + "kerendia_nba_3_0_Rx_05")
df_rx_all = (df_rx_raw
             .withColumn("TREATMENT_CATEGORY", F.regexp_replace("TREATMENT_CATEGORY", r"[\s]*$", ""))
             .filter(F.col(SVC_COL) >= "2019-11-01"))

# Claims written by the HCPs of this run (validation mode: the 10 test HCPs)
df_rx = apply_npi_filter(df_rx_all, NPI_COL)

# All claims of those HCPs' patients, by any HCP (needed for the first-start date)
_rx_patients = df_rx.select("PATIENT_ID").distinct()
df_rx_pat = df_rx_all.join(_rx_patients, on="PATIENT_ID", how="left_semi")

print(f"Rx claims of this run's HCPs: {df_rx.count():,} | patients: {_rx_patients.count():,} | "
      f"all claims of those patients (any HCP): {df_rx_pat.count():,}")

Validation mode — filtered to 10 test NPIs
Rx claims of this run's HCPs: 76,460 | patients: 5,240 | all claims of those patients (any HCP): 269,035


In [0]:
print(f"kerendia_nba_3_0_Rx_05: {len(df_rx_raw.columns)} columns")
display(df_rx_raw.agg(F.count(F.lit(1)).alias("rows"), F.min(SVC_COL).alias("first_date"),
                      F.max(SVC_COL).alias("last_date"), F.countDistinct("PROVIDER_ID").alias("providers")))
display(df_rx.orderBy(NPI_COL, SVC_COL).limit(50))

# One provider and one specialty per test HCP?
display(df_rx.groupBy(NPI_COL, "PROVIDER_ID")
        .agg(F.count(F.lit(1)).alias("claims"), F.countDistinct("PATIENT_ID").alias("patients"),
             F.countDistinct("PRIMARY_SPECIALTY_CODE").alias("specialty_codes"),
             F.first("PRIMARY_SPECIALTY_CODE").alias("specialty"))
        .orderBy(F.desc("claims")))

# Drug classes, and brand groups used by the brand features
display(df_rx.groupBy("TREATMENT_CATEGORY").count().orderBy(F.desc("count")))
display(df_rx.filter(F.col("CATEGORY").isin(_BRANDS)).groupBy("CATEGORY").count().orderBy(F.desc("count")))

kerendia_nba_3_0_Rx_05: 13 columns


rows,first_date,last_date,providers
4032456230,2021-01-02,2026-08-28,231346


PROVIDER_ID,PRIMARY_SPECIALTY_CODE,PATIENT_ID,SVC_DT,PRODUCT_NAME,PRODUCT_STRENGTH,DAYS_SUPPLY,TREATMENT_CATEGORY,CATEGORY,REJECTION_FLAG,REFILL_FLAG,NPI_NUMBER,PULL_DT
29505463,NRP,1305459333,2022-09-07,LISINOPRIL,10 MG,90.000000,ACE/ ARB,ACE/ ARB,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,2770876217,2022-09-07,LEVEMIR FLEXTOUCH,100 UNIT/ML,90.000000,Insulin,Insulin,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,2770876217,2022-09-07,RYBELSUS,14 MG,90.000000,Other- glucose mgmt,GLP-1,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,1197394686,2022-09-07,METOPROLOL TARTRATE,25 MG,90.000000,Calcium channel / Beta blocker,Calcium channel / Beta blocker,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,953716056,2022-09-07,LOSARTAN POTASSIUM,50 MG,90.000000,Combo drug,Combo drug,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,2770876217,2022-09-07,METOPROLOL TARTRATE,25 MG,90.000000,Calcium channel / Beta blocker,Calcium channel / Beta blocker,1,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,2770876217,2022-09-07,VALSARTAN/HYDROCHLOROTHIA,160-12.5 MG,90.000000,ACE/ ARB,ACE/ ARB,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,3383327191,2022-09-08,METFORMIN HYDROCHLORIDE,1000 MG,90.000000,metformin,metformin,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,3383327191,2022-09-08,LISINOPRIL,40 MG,90.000000,ACE/ ARB,ACE/ ARB,0,NEW_PRESCRIPTION,1215675616,2026-09-23
29505463,NRP,3725544030,2022-09-08,GLIPIZIDE ER,2.5 MG/24HR,30.000000,Other- glucose mgmt,Other- glucose mgmt,0,NEW_PRESCRIPTION,1215675616,2026-09-23


NPI_NUMBER,PROVIDER_ID,claims,patients,specialty_codes,specialty
1386651560,7608159,27940,1146,1,FM
1841364841,7812053,20742,705,1,IM
1801062104,10221757,11349,848,1,IM
1508230566,10828538,6103,587,1,NRP
1659887388,22386989,4744,641,1,NRP
1528629052,23915897,1823,270,1,IM
1215675616,29505463,1649,504,1,NRP
1467950774,9069102,873,160,1,NRP
1235697863,23458606,692,187,1,IM
1396374062,24962501,545,192,1,IM


TREATMENT_CATEGORY,count
Calcium channel / Beta blocker,22533
ACE/ ARB,12472
Combo drug,12212
Diuretic_NON_MRA,8468
Other- glucose mgmt,6934
metformin,5694
SGLT2,4531
Insulin,2261
MRA,1355


CATEGORY,count
GLP-1,3425
JARDIANCE,2384
FARXIGA,1563
SPIRONOLACTONE-MRA,1390
DPP-IV,862
INVOKANA 100MG,158
INVOKANA 300MG,32


## 2. What the original builds (claim level, before counting)

The claim month here is the cohort month from `00_cohort_config` (`assign_cohort_month`, offset `OFFSET_DAYS`); at offset 0 it equals the original calendar month (`date_trunc('MM', SVC_DT)`).

### 2.1 First-start dates (original cells 12 and 15)
For every patient and drug class, the date of the patient's **first** claim of that class in the whole table, by any HCP (`df_category_prep`). Same for every patient and brand group, for the 8 brands (`df_brand_prep`).

Because the table starts on 2021-01-02, "first ever" really means "first since 2021-01": patients who were already on a drug before 2021 look like new starts in early 2021. The output keeps months from 2022-01, so 2021 acts as a 12-month burn-in.

Tables below: sample first-start rows, and first-start dates per month (shows the early-2021 spike).

In [0]:
first_cat = (df_rx_pat.groupBy("PATIENT_ID", "TREATMENT_CATEGORY")
             .agg(F.min(SVC_COL).alias("FIRST_SVC_DT")))
first_brand = (df_rx_pat.filter(F.col("CATEGORY").isin(_BRANDS))
               .groupBy("PATIENT_ID", "CATEGORY")
               .agg(F.min(SVC_COL).alias("FIRST_SVC_DT")))

print(f"Patient x drug class first starts: {first_cat.count():,} | patient x brand first starts: {first_brand.count():,}")
display(first_cat.orderBy("PATIENT_ID", "FIRST_SVC_DT").limit(50))
display(first_cat.groupBy(F.date_trunc("MM", "FIRST_SVC_DT").alias("FIRST_START_MONTH")).count()
        .orderBy("FIRST_START_MONTH"))

Patient x drug class first starts: 16,195 | patient x brand first starts: 3,363


PATIENT_ID,TREATMENT_CATEGORY,FIRST_SVC_DT
1000136472,Diuretic_NON_MRA,2021-02-06
1000136472,Calcium channel / Beta blocker,2021-02-15
1000136472,ACE/ ARB,2021-02-15
1000136472,metformin,2025-07-10
1000151945,Other- glucose mgmt,2021-01-19
1000151945,Insulin,2021-01-27
1000151945,Diuretic_NON_MRA,2021-03-28
1000151945,Calcium channel / Beta blocker,2021-04-15
1000151945,Combo drug,2022-10-06
1000151945,metformin,2023-03-07


FIRST_START_MONTH,count
2021-01-01T00:00:00.000Z,2630
2021-02-01T00:00:00.000Z,1261
2021-03-01T00:00:00.000Z,1281
2021-04-01T00:00:00.000Z,520
2021-05-01T00:00:00.000Z,367
2021-06-01T00:00:00.000Z,336
2021-07-01T00:00:00.000Z,293
2021-08-01T00:00:00.000Z,232
2021-09-01T00:00:00.000Z,249
2021-10-01T00:00:00.000Z,234


### 2.2 Claims flagged as a start (original cells 12, 15 and 29)
The HCP's claims are made distinct on (provider, specialty, patient, date, drug class) and joined to the first-start dates on (patient, date, drug class). A claim on the patient's first date gets `NPS_FLAG = 1`; only those are counted. The brand version does the same on (patient, date, brand group).

Points to note:
* If two HCPs write the same patient's first claim on the same day, **both** get the start.
* The count is on distinct (provider, specialty, patient, date, class) rows, so a start is counted once even if the HCP wrote several claims that day.
* `TOTAL_NEW_PATIENT_START` = distinct patients with at least one drug-class start that month (brand starts are not used for it).

Tables below: claims before / after the flag, one patient's claims with the flag (example), and starts per drug class and per brand.

In [0]:
df_rx_m = df_rx.withColumn("SVC_DT_M", assign_cohort_month(F.col(SVC_COL)))

claims_cat = (
    df_rx_m.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "SVC_DT_M").distinct()
    .join(first_cat.withColumnRenamed("FIRST_SVC_DT", SVC_COL).withColumn("NPS_FLAG", F.lit(1)),
          on=["PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY"], how="left")
)
claims_brand = (
    df_rx_m.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "SVC_DT_M", "CATEGORY").distinct()
    .filter(F.col("CATEGORY").isin(_BRANDS))
    .join(first_brand.withColumnRenamed("FIRST_SVC_DT", SVC_COL).withColumn("NPS_FLAG", F.lit(1)),
          on=["PATIENT_ID", SVC_COL, "CATEGORY"], how="left")
)
_starts_cat   = claims_cat.filter("NPS_FLAG = 1")
_starts_brand = claims_brand.filter("NPS_FLAG = 1")
print(f"Drug class: distinct claim rows {claims_cat.count():,} | flagged as start {_starts_cat.count():,}")
print(f"Brand     : distinct claim rows {claims_brand.count():,} | flagged as start {_starts_brand.count():,}")

# Starts credited to more than one HCP (same patient, class and first date) among all claims of these patients
_shared = (df_rx_pat.select("PROVIDER_ID", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY").distinct()
           .join(first_cat.withColumnRenamed("FIRST_SVC_DT", SVC_COL), on=["PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY"], how="inner")
           .groupBy("PATIENT_ID", "TREATMENT_CATEGORY").agg(F.countDistinct("PROVIDER_ID").alias("hcps"))
           .groupBy("hcps").count().withColumnRenamed("count", "patient_class_starts").orderBy("hcps"))
display(_shared)

# Example: the patient with most claims for the first test HCP, with the flag
_pat = (df_rx.groupBy("PATIENT_ID").count().orderBy(F.desc("count")).first()["PATIENT_ID"])
display(claims_cat.filter(F.col("PATIENT_ID") == _pat).orderBy(SVC_COL, "TREATMENT_CATEGORY"))

display(_starts_cat.groupBy("TREATMENT_CATEGORY").count().orderBy(F.desc("count")))
display(_starts_brand.groupBy("CATEGORY").count().orderBy(F.desc("count")))

Drug class: distinct claim rows 73,855 | flagged as start 5,541
Brand     : distinct claim rows 9,804 | flagged as start 892


hcps,patient_class_starts
1,16161
2,34


PATIENT_ID,SVC_DT,TREATMENT_CATEGORY,PROVIDER_ID,PRIMARY_SPECIALTY_CODE,SVC_DT_M,NPS_FLAG
1091553662,2021-01-05,ACE/ ARB,7812053,IM,2020-12-01,1
1091553662,2021-01-05,Calcium channel / Beta blocker,7812053,IM,2020-12-01,1
1091553662,2021-01-05,Other- glucose mgmt,7812053,IM,2020-12-01,1
1091553662,2021-01-05,SGLT2,7812053,IM,2020-12-01,1
1091553662,2021-02-23,ACE/ ARB,7812053,IM,2021-02-01,null
1091553662,2021-02-23,Calcium channel / Beta blocker,7812053,IM,2021-02-01,null
1091553662,2021-02-23,Other- glucose mgmt,7812053,IM,2021-02-01,null
1091553662,2021-02-23,SGLT2,7812053,IM,2021-02-01,null
1091553662,2021-03-05,ACE/ ARB,7812053,IM,2021-02-01,null
1091553662,2021-03-05,Calcium channel / Beta blocker,7812053,IM,2021-02-01,null


TREATMENT_CATEGORY,count
Calcium channel / Beta blocker,1446
ACE/ ARB,976
Combo drug,892
Diuretic_NON_MRA,729
metformin,459
Other- glucose mgmt,399
SGLT2,387
MRA,150
Insulin,103


CATEGORY,count
GLP-1,252
JARDIANCE,237
FARXIGA,171
SPIRONOLACTONE-MRA,151
DPP-IV,67
INVOKANA 100MG,11
INVOKANA 300MG,3


### 2.3 Monthly counts per provider (original cells 12, 17 and 29)
What the original counts per provider per month before filling months and adding windows:
* `TOTAL_PATIENTS_VISIT` = distinct patients with any claim (`df_apt`).
* `TOTAL_PATIENTS_START_<CLASS / BRAND>` = flagged rows per class / brand.
* `TOTAL_NEW_PATIENT_START` = distinct patients with a drug-class start.

The spine is the provider-months with a visit; start columns are left-joined and missing values set to 0. `TOTAL_NEW_PATIENT_START` is built and windowed separately and joined at the end without a fill, so it can be null (the original output has nulls in it).

Table below: these monthly counts for one test HCP.

In [0]:
_visits = (df_rx_m.select("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M").distinct()
           .groupBy("PROVIDER_ID", "SVC_DT_M").agg(F.count(F.lit(1)).alias("TOTAL_PATIENTS_VISIT")))
_cat_m   = (_starts_cat.groupBy("PROVIDER_ID", "SVC_DT_M")
            .agg(F.count(F.lit(1)).alias("CLASS_STARTS"),
                 F.countDistinct("PATIENT_ID").alias("TOTAL_NEW_PATIENT_START")))
_brand_m = _starts_brand.groupBy("PROVIDER_ID", "SVC_DT_M").agg(F.count(F.lit(1)).alias("BRAND_STARTS"))
monthly = (_visits.join(_cat_m, ["PROVIDER_ID", "SVC_DT_M"], "left")
                  .join(_brand_m, ["PROVIDER_ID", "SVC_DT_M"], "left"))

_prov = df_rx.groupBy("PROVIDER_ID").count().orderBy(F.desc("count")).first()["PROVIDER_ID"]
print(f"Example provider: {_prov}")
display(monthly.filter(F.col("PROVIDER_ID") == _prov).orderBy("SVC_DT_M"))

Example provider: 7608159


PROVIDER_ID,SVC_DT_M,TOTAL_PATIENTS_VISIT,CLASS_STARTS,TOTAL_NEW_PATIENT_START,BRAND_STARTS
7608159,2020-12-01,22,29,22,2
7608159,2021-01-01,99,107,89,11
7608159,2021-02-01,148,126,106,14
7608159,2021-03-01,172,136,104,2
7608159,2021-04-01,194,73,67,6
7608159,2021-05-01,211,63,56,3
7608159,2021-06-01,233,56,51,1
7608159,2021-07-01,217,40,32,2
7608159,2021-08-01,215,30,26,2
7608159,2021-09-01,230,20,20,2


## 3. The original output (`kerendia_nba_3_0_claim_rx_nps_monthly`)
After counting, the original fills months without claims (`step2_fill_missing`), adds the last 1/3/6/12-month sums and `RUNNING_` totals (`step3_lag`), `PCT_` shares (each start column / `TOTAL_PATIENTS_VISIT` in the same window), joins `TOTAL_NEW_PATIENT_START` with its own windows, keeps months from 2022-01 and writes one row per provider per month.

Tables below: row and column counts, columns by type, nulls, and a few columns for the test HCPs' providers.

In [0]:
import pandas as pd
df_orig = spark.read.format("delta").load(_NOTEBOOK_OUTPUT + "kerendia_nba_3_0_claim_rx_nps_monthly")
_orig_cols = df_orig.columns

def _kind(c):
    if c in ("PROVIDER_ID", "SVC_DT_M"): return "key"
    if c.startswith("PCT_"):     return "PCT_"
    if c.startswith("RUNNING_"): return "RUNNING_"
    if "_IN_LAST_" in c:         return "_IN_LAST_N_MONTH"
    return "base month"

def _group(c):
    if "NEW_PATIENT_START" in c: return "Total new patients"
    if "PATIENTS_VISIT" in c:   return "Patients visiting"
    if "PATIENTS_START_" in c:  return "Starts per class / brand"
    return "Keys"

_cols = pd.DataFrame({"column": _orig_cols})
_cols["group"] = _cols["column"].map(_group)
_cols["type"]  = _cols["column"].map(_kind)
print(f"kerendia_nba_3_0_claim_rx_nps_monthly: {df_orig.count():,} rows | {len(_orig_cols)} columns")
display(df_orig.agg(F.min("SVC_DT_M").alias("first_month"), F.max("SVC_DT_M").alias("last_month"),
                    F.countDistinct("PROVIDER_ID").alias("providers")))
display(_cols.pivot_table(index="group", columns="type", values="column", aggfunc="count", fill_value=0).reset_index())

# Columns with nulls
_nulls = df_orig.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in _orig_cols]).first().asDict()
display(pd.DataFrame([(c, n) for c, n in _nulls.items() if n], columns=["column", "null_rows"]))

# Test HCPs' providers
_provs = [r[0] for r in df_rx.select("PROVIDER_ID").distinct().collect()]
_show = ["PROVIDER_ID", "SVC_DT_M", "TOTAL_PATIENTS_VISIT", "TOTAL_PATIENTS_START_SGLT2", "TOTAL_PATIENTS_START_JARDIANCE",
         "TOTAL_NEW_PATIENT_START", "TOTAL_PATIENTS_START_SGLT2_IN_LAST_3_MONTH", "RUNNING_TOTAL_PATIENTS_START_SGLT2",
         "PCT_TOTAL_PATIENTS_START_SGLT2", "PCT_TOTAL_PATIENTS_START_SGLT2_IN_LAST_3_MONTH"]
_show = [c for c in _show if c in _orig_cols]
display(df_orig.filter(F.col("PROVIDER_ID").isin(_provs)).select(*_show).orderBy("PROVIDER_ID", "SVC_DT_M").limit(200))
display(_cols)

kerendia_nba_3_0_claim_rx_nps_monthly: 12,955,376 rows | 190 columns


first_month,last_month,providers
2022-01-01T00:00:00.000Z,2026-08-01T00:00:00.000Z,231346


group,PCT_,RUNNING_,_IN_LAST_N_MONTH,base month,key
Keys,0,0,0,0,2
Patients visiting,0,1,4,1,0
Starts per class / brand,80,16,64,16,0
Total new patients,0,1,4,1,0


column,null_rows
TOTAL_PATIENTS_VISIT,1014149
TOTAL_PATIENTS_START_ACE_OR_ARB,1014149
TOTAL_PATIENTS_START_CALCIUM_CHANNEL_OR_BETA_BLOCKER,1014149
TOTAL_PATIENTS_START_COMBO_DRUG,1014149
TOTAL_PATIENTS_START_DIURETIC_NON_MRA,1014149
TOTAL_PATIENTS_START_INSULIN,1014149
TOTAL_PATIENTS_START_MRA,1014149
TOTAL_PATIENTS_START_OTHER_GLUCOSE_MGMT,1014149
TOTAL_PATIENTS_START_SGLT2,1014149
TOTAL_PATIENTS_START_METFORMIN,1014149


PROVIDER_ID,SVC_DT_M,TOTAL_PATIENTS_VISIT,TOTAL_PATIENTS_START_SGLT2,TOTAL_PATIENTS_START_JARDIANCE,TOTAL_NEW_PATIENT_START,TOTAL_PATIENTS_START_SGLT2_IN_LAST_3_MONTH,RUNNING_TOTAL_PATIENTS_START_SGLT2,PCT_TOTAL_PATIENTS_START_SGLT2,PCT_TOTAL_PATIENTS_START_SGLT2_IN_LAST_3_MONTH
10221757,2022-01-01T00:00:00.000Z,119,0,0,12,0,2,0.0,0.0
10221757,2022-02-01T00:00:00.000Z,110,0,0,8,0,2,0.0,0.0
10221757,2022-03-01T00:00:00.000Z,105,0,0,5,0,2,0.0,0.0
10221757,2022-04-01T00:00:00.000Z,108,0,0,11,0,2,0.0,0.0
10221757,2022-05-01T00:00:00.000Z,114,0,0,9,0,2,0.0,0.0
10221757,2022-06-01T00:00:00.000Z,111,0,0,7,0,2,0.0,0.0
10221757,2022-07-01T00:00:00.000Z,65,1,1,3,1,3,0.015384615384615385,0.0034482758620689655
10221757,2022-08-01T00:00:00.000Z,106,4,3,17,5,7,0.03773584905660377,0.01773049645390071
10221757,2022-09-01T00:00:00.000Z,107,2,0,12,7,9,0.018691588785046728,0.025179856115107913
10221757,2022-10-01T00:00:00.000Z,92,1,0,9,7,10,0.010869565217391304,0.022950819672131147


column,group,type
PROVIDER_ID,Keys,key
SVC_DT_M,Keys,key
TOTAL_PATIENTS_VISIT,Patients visiting,base month
TOTAL_PATIENTS_START_ACE_OR_ARB,Starts per class / brand,base month
TOTAL_PATIENTS_START_CALCIUM_CHANNEL_OR_BETA_BLOCKER,Starts per class / brand,base month
TOTAL_PATIENTS_START_COMBO_DRUG,Starts per class / brand,base month
TOTAL_PATIENTS_START_DIURETIC_NON_MRA,Starts per class / brand,base month
TOTAL_PATIENTS_START_INSULIN,Starts per class / brand,base month
TOTAL_PATIENTS_START_MRA,Starts per class / brand,base month
TOTAL_PATIENTS_START_OTHER_GLUCOSE_MGMT,Starts per class / brand,base month


## Walkthrough findings (to confirm with Wenjie before the refactor)

Numbers are for the 10 test HCPs unless stated otherwise.

| # | Finding | Evidence | Impact on the refactor |
| --- | --- | --- | --- |
| 1 | **"New" start is left-censored.** The table starts on 2021-01-02, so a patient's first claim *in the data* counts as a start even if they were already on the drug. | First starts per month: 2,630 in 2021-01, about 1,260 in 2021-02/03, then about 200 per month from 2022. Example provider 7608159: 107 starts in 2021-01 vs about 20 per month in 2022. | The output keeps months from 2022-01 (12-month burn-in). Cohorts from 2025-01 have a 4-year burn-in, so they are not affected. |
| 2 | **Starts are looked up across all HCPs.** The first-start date is computed over every claim of the patient, not only the HCP's own. | 16,195 patient-class starts; 34 of them (0.2%) were credited to 2 HCPs (same first day). | The refactor must compute first-start dates on the full table (or on all claims of the run's patients, as done here in validation). |
| 3 | **`Diuretic + MRA` never appears** as a brand group in the table, so there is no `TOTAL_PATIENTS_START_DIURETIC_AND_MRA` column. | 7 of the 8 brands present; 16 start columns in the output (9 classes + 7 brands). | Keep the same 16 columns (fixed pivot values), so every cohort has the same columns. |
| 4 | **Months with no claims are null, not 0.** `step2_fill_missing` adds every month for every provider, but the main spine is not filled with 0 after it (only `TOTAL_NEW_PATIENT_START` is). | Original output: 1,014,149 rows (7.8%) have null base counts; window and `RUNNING_` columns are null until the provider's first claim month (e.g. 676,937 null `RUNNING_` rows). `PCT_` columns become 0.0. | The cohort output has one row per HCP per cohort, so this only matters for HCPs with no claims in a window. The offset-0 comparison must treat null and 0 as equal; confirm 0 is acceptable. |
| 5 | **`TOTAL_NEW_PATIENT_START` is null for providers with no start ever** (it is left-joined at the end). | 5,824 null rows in the original output. | Same treatment as finding 4. |
| 6 | **`PCT_` uses a sum of monthly visit counts as the denominator.** `TOTAL_PATIENTS_VISIT_IN_LAST_3_MONTH` adds monthly distinct patients, so a patient seen in 3 months counts 3 times. | From `step3_lag` (sum of monthly values). | Keep as is (feature logic unchanged). |
| 7 | **Specialty is part of the distinct key**, so an HCP with two specialty codes would count a start twice. | All 10 test HCPs have a single specialty code. | No impact for the test HCPs; keep as is. |
| 8 | **Header text is stale.** It mentions data from 2016/2017 and a 2019-11 filter; the data starts in 2021-01, so these filters do nothing. | `kerendia_nba_3_0_Rx_05`: 4.03B rows, 2021-01-02 to 2026-08-28, 231,346 providers. | None. |

Next: section 4 (cohort features), which uses the same approach as notebook 04: a long table of (provider, date, feature) rows built once, then one pivot per cohort.

## 4. Cohort features (the refactor)

Same feature logic as the original, rebuilt per cohort. First we build a long feature-row table once; then we pivot those rows into one cohort-wide row per provider.

In [0]:
import re, time
import pandas as pd

NB05_LOOKBACK_WINDOWS = [1, 3, 6, 12]
TOTAL_COL            = "TOTAL_PATIENTS_VISIT"
TOTAL_NEW_COL        = "TOTAL_NEW_PATIENT_START"
START_PREFIX         = "TOTAL_PATIENTS_START_"

# Exact NB05 cleaning from the original notebook (cell 17)
_NB05_CLEAN_STEPS = [(", ", " "), ("/", "_OR_"), ("+", "_AND_"), ("-", "_"), (" ", "_"), ("__", "_")]

def clean_name_05(c):
    for a, b in _NB05_CLEAN_STEPS:
        c = F.regexp_replace(c, re.escape(a), b)
    return F.upper(c)

# Fixed feature list from Wenjie's output: 16 start columns (9 classes + 7 brands; Diuretic + MRA absent)
df_orig = spark.read.format("delta").load(_NOTEBOOK_OUTPUT + "kerendia_nba_3_0_claim_rx_nps_monthly")
START_FEATURES = [
    c for c in df_orig.columns
    if c.startswith(START_PREFIX)
    and "_IN_LAST_" not in c
    and not c.startswith("RUNNING_")
    and not c.startswith("PCT_")
]
FEATURE_VALUES = [TOTAL_COL] + START_FEATURES + [TOTAL_NEW_COL]
PCT_FEATURES   = list(START_FEATURES)


# Brand columns present in the original output (7 in practice)
BRAND_FEATURES = [START_PREFIX + clean_col_name(b) for b in _BRANDS if START_PREFIX + clean_col_name(b) in START_FEATURES]


def build_feature_rows(df_rx_in, df_rx_pat_in, offset_days=None):
    """Long feature table: PROVIDER_ID x event date x feature with integer N."""
    rx = df_rx_in.withColumn("SVC_DT_M", assign_cohort_month(F.col(SVC_COL), offset_days))

    # 1) Monthly distinct patients visiting a provider (specialty kept in the distinct key, as in the original)
    visit_rows = (
        rx.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "SVC_DT_M")
          .groupBy("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", "SVC_DT_M")
          .agg(F.min(SVC_COL).alias(SVC_COL))
          .select("PROVIDER_ID", SVC_COL,
                  F.lit(TOTAL_COL).alias("FEATURE"),
                  F.lit(1).alias("N"))
    )

    # 2) First-start dates across ALL HCPs for the run's patients (all patients in production)
    first_cat = (
        df_rx_pat_in.groupBy("PATIENT_ID", "TREATMENT_CATEGORY")
                   .agg(F.min(SVC_COL).alias("FIRST_SVC_DT"))
    )
    first_brand = (
        df_rx_pat_in.filter(F.col("CATEGORY").isin(_BRANDS))
                   .groupBy("PATIENT_ID", "CATEGORY")
                   .agg(F.min(SVC_COL).alias("FIRST_SVC_DT"))
    )

    # 3) Flagged class starts credited to the provider who wrote the first claim
    class_start_claims = (
        rx.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "SVC_DT_M").distinct()
          .join(first_cat.withColumnRenamed("FIRST_SVC_DT", SVC_COL),
                on=["PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY"], how="inner")
    )
    class_rows = class_start_claims.select(
        "PROVIDER_ID", SVC_COL,
        F.concat(F.lit(START_PREFIX), clean_name_05(F.col("TREATMENT_CATEGORY"))).alias("FEATURE"),
        F.lit(1).alias("N")
    )

    # 4) Same for brand starts (8-brand filter from the original)
    brand_start_claims = (
        rx.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "SVC_DT_M", "CATEGORY").distinct()
          .filter(F.col("CATEGORY").isin(_BRANDS))
          .join(first_brand.withColumnRenamed("FIRST_SVC_DT", SVC_COL),
                on=["PATIENT_ID", SVC_COL, "CATEGORY"], how="inner")
    )
    brand_rows = brand_start_claims.select(
        "PROVIDER_ID", SVC_COL,
        F.concat(F.lit(START_PREFIX), clean_name_05(F.col("CATEGORY"))).alias("FEATURE"),
        F.lit(1).alias("N")
    )

    # 5) Total new patients = distinct patients with at least one class start in that cohort month
    total_new_rows = (
        class_start_claims.groupBy("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M")
                         .agg(F.min(SVC_COL).alias(SVC_COL))
                         .select("PROVIDER_ID", SVC_COL,
                                 F.lit(TOTAL_NEW_COL).alias("FEATURE"),
                                 F.lit(1).alias("N"))
    )

    return (visit_rows
            .unionByName(class_rows)
            .unionByName(brand_rows)
            .unionByName(total_new_rows))


def materialize(df, name):
    """Compute the feature rows once. Validation: keep in memory. Production: write to a scratch Delta path."""
    if get_test_npis() is not None:
        return spark.createDataFrame(df.toPandas(), schema=df.schema)
    path = _NOTEBOOK_OUTPUT + f"_tmp_nb05_feature_rows_{name}"
    df.write.format("delta").mode("overwrite").save(path)
    return spark.read.format("delta").load(path)


# Complete cohorts only (same rule as NB04)
_MAX_SVC_DT   = df_rx_all.agg(F.max(SVC_COL)).first()[0]
COHORT_LIST   = get_cohort_months(max_data_date=_MAX_SVC_DT)
COHORT_BOUNDS = {m: get_cohort_bounds(m) for m in COHORT_LIST}
COHORT_IDS    = [COHORT_BOUNDS[m]["cohort_id"] for m in COHORT_LIST]
PREVIEW_COHORT = COHORT_MONTH if COHORT_MONTH in COHORT_LIST else COHORT_LIST[0]

_t0 = time.time()
df_feat = materialize(build_feature_rows(df_rx, df_rx_pat, offset_days=OFFSET_DAYS), f"offset{OFFSET_DAYS}")
_N_FEAT = df_feat.count()
print(f"Rx data through {_MAX_SVC_DT} -> {len(COHORT_LIST)} complete cohorts: {COHORT_LIST[0]} .. {COHORT_LIST[-1]} | preview cohort: {PREVIEW_COHORT}")
print(f"Feature rows: {_N_FEAT:,} (provider x date x feature) in {time.time() - _t0:,.0f}s")
print(f"Feature columns per window: {len(FEATURE_VALUES)} -> starts: {len(START_FEATURES)}, plus visits + total new patients")

Rx data through 2026-08-28 -> 19 complete cohorts: 2025-01 .. 2026-07 | preview cohort: 2025-02
Feature rows: 55,798 (provider x date x feature) in 92s
Feature columns per window: 18 -> starts: 16, plus visits + total new patients


In [0]:
# ---- Display: the long feature-row table (what gets counted per provider/date/feature) ----
_family = (F.when(F.col("FEATURE") == TOTAL_COL, "A. monthly patients visiting")
           .when(F.col("FEATURE") == TOTAL_NEW_COL, "D. total new patients")
           .when(F.col("FEATURE").isin(BRAND_FEATURES), "C. brand starts")
           .when(F.col("FEATURE").isin(START_FEATURES), "B. drug-class starts")
           .otherwise("other"))

display(df_feat.withColumn("FAMILY", _family)
        .groupBy("FAMILY")
        .agg(F.count(F.lit(1)).alias("rows"),
             F.sum("N").alias("patients_counted"),
             F.countDistinct("FEATURE").alias("feature_columns"))
        .orderBy("FAMILY"))

_ex = df_feat.groupBy("PROVIDER_ID", SVC_COL).agg(F.sum("N").alias("n")).orderBy(F.desc("n")).first()
display(df_feat.filter((F.col("PROVIDER_ID") == _ex["PROVIDER_ID"]) & (F.col(SVC_COL) == _ex[SVC_COL]))
        .withColumn("FAMILY", _family)
        .orderBy("FAMILY", "FEATURE"))

FAMILY,rows,patients_counted,feature_columns
A. monthly patients visiting,44826,44826,1
B. drug-class starts,5541,5541,9
C. brand starts,892,892,7
D. total new patients,4539,4539,1


PROVIDER_ID,SVC_DT,FEATURE,N,FAMILY
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting
7812053,2021-01-07,TOTAL_PATIENTS_VISIT,1,A. monthly patients visiting


In [0]:
# ---- One cohort: windows + RUNNING_ in a single pivot, then PCT_ and metadata ----
def build_cohort(cb, df_feat_in=None, feature_values=None):
    """All features for one cohort, one row per provider with any Rx claim up to the cutoff."""
    L    = df_feat if df_feat_in is None else df_feat_in
    vals = FEATURE_VALUES if feature_values is None else feature_values
    wins = NB05_LOOKBACK_WINDOWS
    cutoff = F.to_date(F.lit(cb["cutoff"].strftime("%Y-%m-%d")))

    # Each row goes to RUNNING_ and to every window whose (start, cutoff] contains its event date
    _suffixes = F.array(F.lit("RUNNING"), *[
        F.when(F.col(SVC_COL) > F.to_date(F.lit(cb["window_bounds"][w]["start"])), F.lit(f"_IN_LAST_{w}_MONTH"))
        for w in wins
    ])

    long = (
        L.filter(F.col(SVC_COL) <= cutoff)
         .withColumn("_suffix", F.explode(_suffixes))
         .filter(F.col("_suffix").isNotNull())
         .withColumn(
             "FEATURE_OUT",
             F.when(F.col("_suffix") == "RUNNING", F.concat(F.lit("RUNNING_"), F.col("FEATURE")))
              .otherwise(F.concat(F.col("FEATURE"), F.col("_suffix")))
         )
    )

    _window_cols  = [f"{f}_IN_LAST_{w}_MONTH" for w in wins for f in vals]
    _running_cols = [f"RUNNING_{f}" for f in vals]
    _pivot_vals   = _window_cols + _running_cols

    wide = (long.groupBy("PROVIDER_ID")
                .pivot("FEATURE_OUT", values=_pivot_vals)
                .agg(F.sum("N")))

    out = wide.na.fill(0)

    pct_exprs = []
    for w in wins:
        denom = f"{TOTAL_COL}_IN_LAST_{w}_MONTH"
        for feat in PCT_FEATURES:
            num = f"{feat}_IN_LAST_{w}_MONTH"
            pct_exprs.append(
                F.when(F.col(denom) != 0, F.col(num) / F.col(denom))
                 .otherwise(F.lit(0.0))
                 .alias(f"PCT_{num}")
            )

    # Same metadata columns as notebooks 02-04
    return (out.select("PROVIDER_ID", *_window_cols, *_running_cols, *pct_exprs)
               .withColumn("COHORT_ID", F.lit(cb["cohort_id"]))
               .withColumn("COHORT_MONTH", F.lit(cb["cohort_month"]))
               .withColumn("OFFSET_DAYS", F.lit(cb["offset_days"]))
               .withColumn("CUTOFF_DATE", F.lit(cb["cutoff"].strftime("%Y-%m-%d")))
               .withColumn("SVC_DT_M", F.to_timestamp(F.lit(f"{cb['cohort_month']}-01")))
               .withColumn("PULL_DT", F.current_date()))


_preview_cb = COHORT_BOUNDS[PREVIEW_COHORT]
_t1 = time.time()
claims_preview = build_cohort(_preview_cb)
_pv_cols = claims_preview.columns
print(f"Preview {_preview_cb['cohort_id']} (cutoff {_preview_cb['cutoff']:%Y-%m-%d}) | providers: {claims_preview.count():,} | columns: {len(_pv_cols):,} | {time.time() - _t1:,.0f}s")

Preview 2025-02_offset7 (cutoff 2025-03-06) | providers: 10 | columns: 161 | 2s


In [0]:
# ---- Display: preview cohort, keys + totals + example columns ----
_ex_cols = [c for c in [
    f"{TOTAL_COL}_IN_LAST_1_MONTH", f"{TOTAL_COL}_IN_LAST_3_MONTH", f"{TOTAL_COL}_IN_LAST_12_MONTH", f"RUNNING_{TOTAL_COL}",
    "TOTAL_PATIENTS_START_SGLT2_IN_LAST_12_MONTH",
    "TOTAL_PATIENTS_START_JARDIANCE_IN_LAST_12_MONTH",
    "TOTAL_NEW_PATIENT_START_IN_LAST_12_MONTH",
    "RUNNING_TOTAL_NEW_PATIENT_START",
    "PCT_TOTAL_PATIENTS_START_SGLT2_IN_LAST_12_MONTH",
    "PCT_TOTAL_PATIENTS_START_JARDIANCE_IN_LAST_12_MONTH",
] if c in _pv_cols]

display(claims_preview.select("PROVIDER_ID", "COHORT_ID", "CUTOFF_DATE", *_ex_cols).orderBy("PROVIDER_ID"))

PROVIDER_ID,COHORT_ID,CUTOFF_DATE,TOTAL_PATIENTS_VISIT_IN_LAST_1_MONTH,TOTAL_PATIENTS_VISIT_IN_LAST_3_MONTH,TOTAL_PATIENTS_VISIT_IN_LAST_12_MONTH,RUNNING_TOTAL_PATIENTS_VISIT,TOTAL_PATIENTS_START_SGLT2_IN_LAST_12_MONTH,TOTAL_PATIENTS_START_JARDIANCE_IN_LAST_12_MONTH,TOTAL_NEW_PATIENT_START_IN_LAST_12_MONTH,RUNNING_TOTAL_NEW_PATIENT_START,PCT_TOTAL_PATIENTS_START_SGLT2_IN_LAST_12_MONTH,PCT_TOTAL_PATIENTS_START_JARDIANCE_IN_LAST_12_MONTH
10221757,2025-02_offset7,2025-03-06,113,370,1536,5755,6,4,122,800,0.00390625,0.0026041666666666665
10828538,2025-02_offset7,2025-03-06,43,141,571,3126,1,1,78,537,0.0017513134851138354,0.0017513134851138354
22386989,2025-02_offset7,2025-03-06,84,233,704,1828,5,2,43,133,0.007102272727272727,0.002840909090909091
23458606,2025-02_offset7,2025-03-06,9,23,33,157,0,0,10,39,0.0,0.0
23915897,2025-02_offset7,2025-03-06,15,46,413,1242,0,0,18,171,0.0,0.0
24962501,2025-02_offset7,2025-03-06,0,5,26,278,3,3,13,76,0.11538461538461539,0.11538461538461539
29505463,2025-02_offset7,2025-03-06,0,0,48,963,0,0,6,115,0.0,0.0
7608159,2025-02_offset7,2025-03-06,220,763,3235,12096,13,7,185,1199,0.00401854714064915,0.0021638330757341576
7812053,2025-02_offset7,2025-03-06,184,496,2033,7137,8,6,57,784,0.003935071323167732,0.002951303492375799
9069102,2025-02_offset7,2025-03-06,12,29,104,406,0,0,3,31,0.0,0.0


## 5. Build and Write All Cohorts
Runs `build_cohort` for every cohort in `COHORT_LIST` and writes one `COHORT_ID` partition per cohort (`replaceWhere`, so other cohorts and offsets are kept) to `claim_rx_nps_monthly_cohort`. The input already carries `PROVIDER_ID`, so this is the provider-level output directly (no NPI mapping step). Wenjie's `kerendia_nba_3_0_claim_rx_nps_monthly` is not touched.

Outputs below: one line per cohort written, then the completeness checks (all cohorts present, one row per provider per cohort, no nulls, same columns in every cohort) and the totals per provider and cohort.

In [0]:
OUTPUT_PATH = _NOTEBOOK_OUTPUT + "claim_rx_nps_monthly_cohort"

_t_all = time.time()
for m in COHORT_LIST:
    cb = COHORT_BOUNDS[m]
    _t0 = time.time()
    build_cohort(cb).write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("replaceWhere", f"COHORT_ID = '{cb['cohort_id']}'") \
        .partitionBy("COHORT_ID") \
        .save(OUTPUT_PATH)
    print(f"{cb['cohort_id']}: written (cutoff {cb['cutoff']:%Y-%m-%d}) in {time.time() - _t0:,.0f}s")

print(f"\n{len(COHORT_LIST)} cohorts written to {OUTPUT_PATH} in {(time.time() - _t_all) / 60:,.1f} min")

2025-01_offset7: written (cutoff 2025-02-06) in 4s
2025-02_offset7: written (cutoff 2025-03-06) in 2s
2025-03_offset7: written (cutoff 2025-04-06) in 2s
2025-04_offset7: written (cutoff 2025-05-06) in 2s
2025-05_offset7: written (cutoff 2025-06-06) in 2s
2025-06_offset7: written (cutoff 2025-07-06) in 2s
2025-07_offset7: written (cutoff 2025-08-06) in 2s
2025-08_offset7: written (cutoff 2025-09-06) in 2s
2025-09_offset7: written (cutoff 2025-10-06) in 2s
2025-10_offset7: written (cutoff 2025-11-06) in 2s
2025-11_offset7: written (cutoff 2025-12-06) in 2s
2025-12_offset7: written (cutoff 2026-01-06) in 2s
2026-01_offset7: written (cutoff 2026-02-06) in 2s
2026-02_offset7: written (cutoff 2026-03-06) in 2s
2026-03_offset7: written (cutoff 2026-04-06) in 2s
2026-04_offset7: written (cutoff 2026-05-06) in 2s
2026-05_offset7: written (cutoff 2026-06-06) in 2s
2026-06_offset7: written (cutoff 2026-07-06) in 2s
2026-07_offset7: written (cutoff 2026-08-06) in 2s

19 cohorts written to /Volumes

In [0]:
# Read the written table directly (independent of the preview variables)
df_all_cohorts = spark.read.format("delta").load(OUTPUT_PATH)

_ids_in_table  = sorted(r[0] for r in df_all_cohorts.select("COHORT_ID").distinct().collect())
_missing       = sorted(set(COHORT_IDS) - set(_ids_in_table))
_extra         = sorted(set(_ids_in_table) - set(COHORT_IDS))
df_all_cohorts = df_all_cohorts.filter(F.col("COHORT_ID").isin(COHORT_IDS))
_n_rows  = df_all_cohorts.count()
_n_keys  = df_all_cohorts.select("PROVIDER_ID", "COHORT_ID").distinct().count()
_n_provs = df_all_cohorts.select("PROVIDER_ID").distinct().count()
_nulls   = df_all_cohorts.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_all_cohorts.columns]).first()
_null_cells = builtins.sum(v or 0 for v in _nulls)

# Expected columns: 18 features x 4 windows + 18 RUNNING_ + 16 x 4 PCT_ + 7 metadata (incl. PROVIDER_ID)
_exp_cols = (len(FEATURE_VALUES) * len(NB05_LOOKBACK_WINDOWS) + len(FEATURE_VALUES)
             + len(PCT_FEATURES) * len(NB05_LOOKBACK_WINDOWS) + 7)

print(f"Cohorts expected: {len(COHORT_IDS)} | in table: {len(_ids_in_table)} | "
      f"missing: {_missing or 'none'} | extra (not in this run): {_extra or 'none'}")
print(f"Rows: {_n_rows:,} | distinct (PROVIDER_ID, COHORT_ID): {_n_keys:,} | duplicates: {_n_rows - _n_keys} | "
      f"providers: {_n_provs} | columns: {len(df_all_cohorts.columns):,} (expected {_exp_cols}) | null cells: {_null_cells:,}")
display(df_all_cohorts.groupBy("COHORT_ID", "CUTOFF_DATE").count().orderBy("COHORT_ID"))

_key_cols = ["PROVIDER_ID", "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT"]
_tot_cols = ([f"{TOTAL_COL}_IN_LAST_{w}_MONTH" for w in NB05_LOOKBACK_WINDOWS] + [f"RUNNING_{TOTAL_COL}"]
             + [f"{TOTAL_NEW_COL}_IN_LAST_{w}_MONTH" for w in NB05_LOOKBACK_WINDOWS] + [f"RUNNING_{TOTAL_NEW_COL}"])
display(df_all_cohorts.select(*_key_cols, *_tot_cols).orderBy("PROVIDER_ID", "COHORT_ID"))

Cohorts expected: 19 | in table: 19 | missing: none | extra (not in this run): none
Rows: 190 | distinct (PROVIDER_ID, COHORT_ID): 190 | duplicates: 0 | providers: 10 | columns: 161 (expected 161) | null cells: 0


COHORT_ID,CUTOFF_DATE,count
2025-01_offset7,2025-02-06,10
2025-02_offset7,2025-03-06,10
2025-03_offset7,2025-04-06,10
2025-04_offset7,2025-05-06,10
2025-05_offset7,2025-06-06,10
2025-06_offset7,2025-07-06,10
2025-07_offset7,2025-08-06,10
2025-08_offset7,2025-09-06,10
2025-09_offset7,2025-10-06,10
2025-10_offset7,2025-11-06,10


PROVIDER_ID,COHORT_ID,COHORT_MONTH,OFFSET_DAYS,CUTOFF_DATE,SVC_DT_M,PULL_DT,TOTAL_PATIENTS_VISIT_IN_LAST_1_MONTH,TOTAL_PATIENTS_VISIT_IN_LAST_3_MONTH,TOTAL_PATIENTS_VISIT_IN_LAST_6_MONTH,TOTAL_PATIENTS_VISIT_IN_LAST_12_MONTH,RUNNING_TOTAL_PATIENTS_VISIT,TOTAL_NEW_PATIENT_START_IN_LAST_1_MONTH,TOTAL_NEW_PATIENT_START_IN_LAST_3_MONTH,TOTAL_NEW_PATIENT_START_IN_LAST_6_MONTH,TOTAL_NEW_PATIENT_START_IN_LAST_12_MONTH,RUNNING_TOTAL_NEW_PATIENT_START
10221757,2025-01_offset7,2025-01,7,2025-02-06,2025-01-01T00:00:00.000Z,2026-10-22,144,381,792,1548,5642,12,23,54,118,788
10221757,2025-02_offset7,2025-02,7,2025-03-06,2025-02-01T00:00:00.000Z,2026-10-22,113,370,782,1536,5755,12,29,58,122,800
10221757,2025-03_offset7,2025-03,7,2025-04-06,2025-03-01T00:00:00.000Z,2026-10-22,141,398,782,1552,5896,16,40,61,125,816
10221757,2025-04_offset7,2025-04,7,2025-05-06,2025-04-01T00:00:00.000Z,2026-10-22,126,380,761,1549,6022,9,37,60,121,825
10221757,2025-05_offset7,2025-05,7,2025-06-06,2025-05-01T00:00:00.000Z,2026-10-22,123,390,760,1556,6145,9,34,63,116,834
10221757,2025-06_offset7,2025-06,7,2025-07-06,2025-06-01T00:00:00.000Z,2026-10-22,134,383,781,1569,6279,9,27,67,116,843
10221757,2025-07_offset7,2025-07,7,2025-08-06,2025-07-01T00:00:00.000Z,2026-10-22,132,389,769,1561,6411,13,31,68,122,856
10221757,2025-08_offset7,2025-08,7,2025-09-06,2025-08-01T00:00:00.000Z,2026-10-22,127,393,783,1565,6538,2,24,58,116,858
10221757,2025-09_offset7,2025-09,7,2025-10-06,2025-09-01T00:00:00.000Z,2026-10-22,146,405,788,1570,6684,14,29,56,117,872
10221757,2025-10_offset7,2025-10,7,2025-11-06,2025-10-01T00:00:00.000Z,2026-10-22,141,414,803,1564,6825,17,33,64,124,889


## 6. Validation

1. **Offset = 0 comparison with the original (every cohort):** rebuild the feature rows at offset 0 (calendar months) for the test HCPs and compare every column with Wenjie's `kerendia_nba_3_0_claim_rx_nps_monthly`, cohort month vs original month. Original base-month columns (including base-month `PCT_`) are compared with our `_IN_LAST_1_MONTH` columns; original nulls count as 0 (findings 4 and 5). Providers that are only in the original for a month (no claims yet) must be all zeros.
2. **HCP trace:** for the busiest test HCP, rebuild every feature in pandas from the raw table (separate code: pandas first-start dates, Python name cleaning, windows counted by cohort month instead of dates) and compare every window, `RUNNING_` and `PCT_` value with the written table, for every cohort.

Outputs below: one summary row per month / cohort (cells compared, mismatches) and a mismatch table, which should be empty.

In [0]:
import numpy as np

# ---- Validation 1: offset = 0 parity vs original kerendia_nba_3_0_claim_rx_nps_monthly, every cohort ----
_npis = get_test_npis()
if _npis is None:
    print("Production mode: parity check skipped (runs on the test HCPs only)")
else:
    _t_par = time.time()
    df_feat0 = materialize(build_feature_rows(df_rx, df_rx_pat, offset_days=0), "offset0")
    print(f"Offset-0 feature rows: {df_feat0.count():,} | {time.time() - _t_par:,.0f}s")

    _provs_test = [r[0] for r in df_rx.select("PROVIDER_ID").distinct().collect()]
    _orig_all = (df_orig.filter(F.col("PROVIDER_ID").isin(_provs_test))
                 .withColumn("_M", F.date_format("SVC_DT_M", "yyyy-MM")))
    _orig_months  = {r[0] for r in _orig_all.select("_M").distinct().collect()}
    PARITY_MONTHS = [m for m in get_cohort_months(max_data_date=_MAX_SVC_DT, offset_days=0) if m in _orig_months]
    pd_orig_all = _orig_all.filter(F.col("_M").isin(PARITY_MONTHS)).toPandas()
    print(f"Original rows for the test providers: {len(pd_orig_all):,} | months compared: {len(PARITY_MONTHS)}")

    _skip = {"SVC_DT_M", "PULL_DT", "_M", "PROVIDER_ID"}
    _orig_cmp_cols = [c for c in pd_orig_all.columns if c not in _skip]
    # Original column -> cohort-based column (base-month columns map to _IN_LAST_1_MONTH)
    _map = {oc: (oc if ("_IN_LAST_" in oc or oc.startswith("RUNNING_")) else f"{oc}_IN_LAST_1_MONTH")
            for oc in _orig_cmp_cols}
    _meta = {"COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT", "PROVIDER_ID"}

    def _num(df):
        return df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)

    summary, mismatches = [], []
    for m in PARITY_MONTHS:
        cb0 = get_cohort_bounds(m, offset_days=0)
        pd_new  = build_cohort(cb0, df_feat0).toPandas().set_index("PROVIDER_ID")
        pd_orig = pd_orig_all[pd_orig_all["_M"] == m].set_index("PROVIDER_ID")

        common    = sorted(set(pd_new.index) & set(pd_orig.index))
        only_orig = sorted(set(pd_orig.index) - set(pd_new.index))
        only_new  = sorted(set(pd_new.index) - set(pd_orig.index))
        present   = [oc for oc in _orig_cmp_cols if _map[oc] in pd_new.columns]
        absent    = [oc for oc in _orig_cmp_cols if _map[oc] not in pd_new.columns]
        extra_new = [c for c in pd_new.columns if c not in _meta and c not in set(_map.values())]

        o = _num(pd_orig.loc[common, present])
        n = _num(pd_new.loc[common, [_map[oc] for oc in present]])
        bad = ~np.isclose(o.values, n.values, rtol=1e-9, atol=1e-9)
        for i, j in list(zip(*np.where(bad)))[:50]:
            mismatches.append({"cohort": cb0["cohort_id"], "PROVIDER_ID": common[i], "original_col": present[j],
                               "new_col": _map[present[j]], "original": o.values[i, j], "new": n.values[i, j]})
        # Providers only in the original that month (no claims yet) must be all zeros
        only_orig_nonzero = (int((_num(pd_orig.loc[only_orig, _orig_cmp_cols]) != 0).any(axis=1).sum())
                             if only_orig else 0)

        summary.append({"cohort_offset0": cb0["cohort_id"], "cutoff": f"{cb0['cutoff']:%Y-%m-%d}",
                        "providers_compared": len(common), "only_in_original": len(only_orig),
                        "only_in_original_nonzero": only_orig_nonzero, "only_in_new": len(only_new),
                        "columns_compared": len(present), "original_cols_missing": len(absent),
                        "new_cols_not_in_original": len(extra_new),
                        "cells_compared": int(o.size), "mismatches": int(bad.sum())})

    df_parity = pd.DataFrame(summary)
    _tot_mis = int(df_parity["mismatches"].sum())
    _ok = (_tot_mis == 0 and df_parity["only_in_original_nonzero"].sum() == 0 and df_parity["only_in_new"].sum() == 0
           and df_parity["original_cols_missing"].sum() == 0)
    print(f"Parity {'PASSED' if _ok else 'FAILED'}: {len(PARITY_MONTHS)} months | "
          f"cells compared: {int(df_parity['cells_compared'].sum()):,} | mismatches: {_tot_mis:,} | "
          f"{time.time() - _t_par:,.0f}s")
    display(df_parity)
    if mismatches:
        display(pd.DataFrame(mismatches))
    else:
        print("Mismatch table: empty (no differing cells)")

Offset-0 feature rows: 55,754 | 306s
Original rows for the test providers: 190 | months compared: 19
Parity PASSED: 19 months | cells compared: 35,720 | mismatches: 0 | 365s


cohort_offset0,cutoff,providers_compared,only_in_original,only_in_original_nonzero,only_in_new,columns_compared,original_cols_missing,new_cols_not_in_original,cells_compared,mismatches
2025-01_offset0,2025-01-31,10,0,0,0,188,0,0,1880,0
2025-02_offset0,2025-02-28,10,0,0,0,188,0,0,1880,0
2025-03_offset0,2025-03-31,10,0,0,0,188,0,0,1880,0
2025-04_offset0,2025-04-30,10,0,0,0,188,0,0,1880,0
2025-05_offset0,2025-05-31,10,0,0,0,188,0,0,1880,0
2025-06_offset0,2025-06-30,10,0,0,0,188,0,0,1880,0
2025-07_offset0,2025-07-31,10,0,0,0,188,0,0,1880,0
2025-08_offset0,2025-08-31,10,0,0,0,188,0,0,1880,0
2025-09_offset0,2025-09-30,10,0,0,0,188,0,0,1880,0
2025-10_offset0,2025-10-31,10,0,0,0,188,0,0,1880,0


Mismatch table: empty (no differing cells)


In [0]:
# ---- Validation 2: HCP trace. Every feature rebuilt in pandas from the raw table vs written table ----
_trace_row = df_rx.groupBy(NPI_COL, "PROVIDER_ID").count().orderBy(F.desc("count")).first()
TRACE_NPI, TRACE_PROV = _trace_row[NPI_COL], _trace_row["PROVIDER_ID"]
_off = OFFSET_DAYS
_t_tr = time.time()

def _pm(s):
    """Cohort month in pandas (the offset-th day starts the month)."""
    return ((s - pd.Timedelta(days=_off - 1)) if _off > 0 else s).dt.to_period("M")

def _py_clean(s):
    for a, b in [(", ", " "), ("/", "_OR_"), ("+", "_AND_"), ("-", "_"), (" ", "_"), ("__", "_")]:
        s = s.replace(a, b)
    return s.upper()

_cols_needed = ["PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "CATEGORY"]
# Raw table, own cleaning (trailing spaces) and own date filter
pdx = df_rx_raw.filter(F.col(NPI_COL) == TRACE_NPI).select(*_cols_needed).toPandas()
_tp = pdx["PATIENT_ID"].dropna().unique().tolist()
pall = (df_rx_raw.filter(F.col("PATIENT_ID").isin(_tp))
        .select("PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "CATEGORY").toPandas())
for _d in (pdx, pall):
    _d[SVC_COL] = pd.to_datetime(_d[SVC_COL])
    _d["TREATMENT_CATEGORY"] = _d["TREATMENT_CATEGORY"].str.rstrip()
pdx  = pdx[pdx[SVC_COL] >= "2019-11-01"]
pall = pall[pall[SVC_COL] >= "2019-11-01"]
pdx["M"] = _pm(pdx[SVC_COL])

# First dates per patient x class and patient x brand, across all HCPs
_first_cat   = pall.groupby(["PATIENT_ID", "TREATMENT_CATEGORY"])[SVC_COL].min().rename("FIRST").reset_index()
_first_brand = (pall[pall["CATEGORY"].isin(_BRANDS)]
                .groupby(["PATIENT_ID", "CATEGORY"])[SVC_COL].min().rename("FIRST").reset_index())

# Events (FEATURE, cohort month), one per +1
_visits = pdx[["PRIMARY_SPECIALTY_CODE", "PATIENT_ID", "M"]].drop_duplicates().assign(FEATURE=TOTAL_COL)
_cs = (pdx[["PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "M"]].drop_duplicates()
       .merge(_first_cat, on=["PATIENT_ID", "TREATMENT_CATEGORY"]))
_cs = _cs[_cs[SVC_COL] == _cs["FIRST"]].copy()
_cs["FEATURE"] = START_PREFIX + _cs["TREATMENT_CATEGORY"].map(_py_clean)
_bs = (pdx[pdx["CATEGORY"].isin(_BRANDS)][["PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "CATEGORY", "M"]].drop_duplicates()
       .merge(_first_brand, on=["PATIENT_ID", "CATEGORY"]))
_bs = _bs[_bs[SVC_COL] == _bs["FIRST"]].copy()
_bs["FEATURE"] = START_PREFIX + _bs["CATEGORY"].map(_py_clean)
_tn = _cs[["PATIENT_ID", "M"]].drop_duplicates().assign(FEATURE=TOTAL_NEW_COL)
_events = pd.concat([d[["FEATURE", "M"]] for d in (_visits, _cs, _bs, _tn)], ignore_index=True)
_counts = _events.groupby(["FEATURE", "M"]).size()

_written = (spark.read.format("delta").load(OUTPUT_PATH)
            .filter((F.col("PROVIDER_ID") == TRACE_PROV) & F.col("COHORT_ID").isin(COHORT_IDS)).toPandas()
            .set_index("COHORT_ID"))

trace_rows, trace_bad = [], []
for m in COHORT_LIST:
    cb = COHORT_BOUNDS[m]
    Mc = pd.Period(m, "M").ordinal
    exp = {}
    for f in FEATURE_VALUES:
        s = _counts.xs(f, level="FEATURE") if f in _counts.index.get_level_values("FEATURE") else pd.Series(dtype=int)
        mi = np.array([p.ordinal for p in s.index], dtype=int)  # month as integer ordinal
        v  = s.to_numpy()
        exp[f"RUNNING_{f}"] = int(v[mi <= Mc].sum())
        for w in NB05_LOOKBACK_WINDOWS:
            exp[f"{f}_IN_LAST_{w}_MONTH"] = int(v[(mi > Mc - w) & (mi <= Mc)].sum())
    for w in NB05_LOOKBACK_WINDOWS:
        den = exp[f"{TOTAL_COL}_IN_LAST_{w}_MONTH"]
        for f in PCT_FEATURES:
            exp[f"PCT_{f}_IN_LAST_{w}_MONTH"] = (exp[f"{f}_IN_LAST_{w}_MONTH"] / den) if den else 0.0
    row = _written.loc[cb["cohort_id"]] if cb["cohort_id"] in _written.index else None
    n_bad = 0
    for c, v in exp.items():
        got = float(row[c]) if row is not None and c in row.index else None
        if got is None or not np.isclose(got, v, rtol=1e-9, atol=1e-9):
            n_bad += 1
            if len(trace_bad) < 50:
                trace_bad.append({"cohort": cb["cohort_id"], "column": c, "expected": v, "written": got})
    trace_rows.append({"cohort": cb["cohort_id"], "cutoff": f"{cb['cutoff']:%Y-%m-%d}", "checks": len(exp),
                       "mismatches": n_bad,
                       "visits_12m": exp[f"{TOTAL_COL}_IN_LAST_12_MONTH"],
                       "new_patients_12m": exp[f"{TOTAL_NEW_COL}_IN_LAST_12_MONTH"]})

df_trace = pd.DataFrame(trace_rows)
_n_checks, _n_bad = int(df_trace["checks"].sum()), int(df_trace["mismatches"].sum())
print(f"HCP trace {'PASSED' if _n_bad == 0 else 'FAILED'}: NPI {TRACE_NPI} / provider {TRACE_PROV} | "
      f"claims {len(pdx):,} | patients {len(_tp):,} | all claims of those patients {len(pall):,} | "
      f"cohorts {len(COHORT_LIST)} | checks {_n_checks:,} | mismatches {_n_bad:,} | {time.time() - _t_tr:,.0f}s")
display(df_trace)
if trace_bad:
    display(pd.DataFrame(trace_bad))
else:
    print("Mismatch table: empty (no differing values)")

HCP trace PASSED: NPI 1386651560 / provider 7608159 | claims 27,940 | patients 1,146 | all claims of those patients 49,437 | cohorts 19 | checks 2,926 | mismatches 0 | 12s


cohort,cutoff,checks,mismatches,visits_12m,new_patients_12m
2025-01_offset7,2025-02-06,154,0,3271,200
2025-02_offset7,2025-03-06,154,0,3235,185
2025-03_offset7,2025-04-06,154,0,3241,171
2025-04_offset7,2025-05-06,154,0,3234,161
2025-05_offset7,2025-06-06,154,0,3184,148
2025-06_offset7,2025-07-06,154,0,3148,137
2025-07_offset7,2025-08-06,154,0,3125,124
2025-08_offset7,2025-09-06,154,0,3053,118
2025-09_offset7,2025-10-06,154,0,3032,123
2025-10_offset7,2025-11-06,154,0,2991,112


Mismatch table: empty (no differing values)
